# Big Model Training, Quantization, Conversion and Optuna Optimization

## Set up

In [ ]:
import sys

COLAB = "google.colab" in sys.modules

# Environment Setup
if COLAB:
    print("Running in Google Colab. Setting up environment...")
    BASE_DIR = './'  # In Colab, everything stays inside /content/

    import gdown
    requirements_ID = '1XHtGbTN-NEHnvAyM7ufYi4mw0K2lN9NN'
    gdown.download(id=requirements_ID, output='requirements.txt', quiet=False)
    %pip install -r "requirements.txt"
else:
    print("Running locally. Assuming files are already present.")
    BASE_DIR = '../' # Locally, everything is one level up from /src

    requirements_path = f"{BASE_DIR}requirements.txt"
    %pip install -r "{requirements_path}"

In [ ]:
import os
import tensorflow as tf
import tf_keras as keras
import numpy as np
from sklearn.model_selection import train_test_split
import optuna
from tf_keras.models import Sequential
from tf_keras.layers import Conv2D, Flatten, Dense, ReLU, Input, Dropout
from tf_keras.optimizers import Adam
from tf_keras.callbacks import EarlyStopping, ModelCheckpoint, ReduceLROnPlateau, TensorBoard
from tf_keras.losses import SparseCategoricalCrossentropy
from quantizeml import load_model
from quantizeml.models import quantize, QuantizationParams
from quantizeml.models.quantize import dump_config
from cnn2snn import set_akida_version, AkidaVersion, convert
import cnn2snn
import akida
import json
import datetime
import matplotlib.pyplot as plt

In [ ]:
# Force TensorFlow to use the legacy Keras 2 backend
# This is strictly required because Brainchip's cnn2snn converter crashes with Keras 3
os.environ["TF_USE_LEGACY_KERAS"] = "1"

print(f"TensorFlow version: {tf.__version__}")
print(f"Keras module: {keras.__name__}")
print(f"Keras version: {keras.__version__}")

## Dataset download and load

In [ ]:
# Dataset download
dataset_dir = os.path.join(BASE_DIR, 'data/processed_data/')
dataset_path = os.path.join(dataset_dir, 'X_watch_20Hz_scaled_big.npy')
labels_path = os.path.join(dataset_dir, 'y_watch_labels_big.npy')

if not os.path.exists(dataset_path) or not os.path.exists(labels_path):
    dataset_ID = '1J0j7FzpsfXdbFFaTqgIH1XNjQT31SQDC'
    labels_ID = '1QK7vxsYoPedcbsWKziA4NOp9x4SMVqH4'

    print(f"Dataset or labels not found in {dataset_dir}. Downloading...")
    os.makedirs(dataset_dir, exist_ok=True)
        
    gdown.download(id=dataset_ID, output=dataset_path, quiet=False)
    gdown.download(id=labels_ID, output=labels_path, quiet=False)
    print("Download complete!")
else:
    print(f"Dataset and labels already exist in {dataset_dir}. Skipping download.")

In [ ]:
# Dataset load
try:
    X_data = np.load(dataset_path)
    y_labels = np.load(labels_path).astype(np.int32)

    print(f"Dataset loaded successfully from: {dataset_path}")
    print(f"Labels loaded successfully from: {labels_path}")

    print(f"\nX_data shape: {X_data.shape}")
    print(f"y_labels shape: {y_labels.shape}, dtype: {y_labels.dtype}")

except FileNotFoundError:
    print(f"Error: One or both files were not found.")
except Exception as e:
    print(f"An error occurred: {e}")

### Dataset Split

In [ ]:
def split_and_save_dataset(X, y, train_size=0.7, val_size=0.15, test_size=0.15, random_state=42):
    X_train, X_temp, y_train, y_temp = train_test_split(
        X, y, train_size=train_size, random_state=random_state, stratify=y
    )

    relative_val_size = val_size / (val_size + test_size)
    X_val, X_test, y_val, y_test = train_test_split(
        X_temp, y_temp, train_size=relative_val_size, random_state=random_state, stratify=y_temp
    )

    return X_train, X_val, X_test, y_train, y_val, y_test

In [ ]:
X_train, X_val, X_test, y_train, y_val, y_test = split_and_save_dataset(X_data, y_labels, train_size=0.7, val_size=0.15, test_size=0.15)

# Remap the labels into a continuous range [0, N-1]
unique_labels = np.unique(y_train)
label_mapping = {old_label: new_label for new_label, old_label in enumerate(unique_labels)}

def apply_mapping(labels, mapping):
    return np.array([mapping[label] for label in labels], dtype=np.int32)

y_train = apply_mapping(y_train, label_mapping)
y_val = apply_mapping(y_val, label_mapping)
y_test = apply_mapping(y_test, label_mapping)

num_classes = len(unique_labels)

print(f"Number of unique classes detected: {num_classes}")
print(f"Applied mapping: {label_mapping}")
print(f"X_train dimensions: {X_train.shape}, y_train: {y_train.shape}")

In [ ]:
X_train_reshaped = X_train.reshape(-1, 40, 6, 1)
X_val_reshaped = X_val.reshape(-1, 40, 6, 1)
X_test_reshaped = X_test.reshape(-1, 40, 6, 1)

# 41x9 for hardware symmetric padding
pad_config = ((0,0), (0,1), (1,2), (0,0))
X_train_padded = np.pad(X_train_reshaped, pad_width=pad_config, mode='constant', constant_values=0)
X_val_padded = np.pad(X_val_reshaped, pad_width=pad_config, mode='constant', constant_values=0)
X_test_padded = np.pad(X_test_reshaped, pad_width=pad_config, mode='constant', constant_values=0)

x_min = np.min(X_train_padded)
x_max = np.max(X_train_padded)

# Range [0.0, 1.0] in float32 (for input_quantizer)
X_train = ((X_train_padded - x_min) / (x_max - x_min + 1e-8)).astype(np.float32)
X_val = ((X_val_padded - x_min) / (x_max - x_min + 1e-8)).astype(np.float32)
X_test = ((X_test_padded - x_min) / (x_max - x_min + 1e-8)).astype(np.float32)

## Model (Big) and Optuna
We use Optuna to find the optimal neural network configuration. The model will be trained on the entire dataset.

In [ ]:
# Dictionaries for fixed parameters
hardware_settings = {
    "input_shape": (41, 9, 1),
    "num_classes": num_classes,
    "max_relu": 6,
    "qparams": QuantizationParams(input_weight_bits=8, weight_bits=4, activation_bits=4)
}

training_settings = {
    "epochs": 10,
    "patience": 5
}

fixed_settings = dict(hardware_settings, **training_settings)

In [ ]:
def create_dynamic_model(trial, settings):
    """Builds the Keras model based on the parameters suggested by Optuna."""

    # Hyperparameters chosen by Optuna
    conv1_filters = trial.suggest_categorical('conv1_filters', [32, 48, 64])
    conv2_filters = trial.suggest_categorical('conv2_filters', [64, 96, 128])
    conv3_filters = trial.suggest_categorical('conv3_filters', [128, 192, 256])
    conv4_filters = trial.suggest_categorical('conv4_filters', [16, 24, 32, 48, 64])

    # Optuna decides whether to use 1 or 2 final Dense layers
    num_dense_layers = trial.suggest_int('num_dense_layers', 1, 2)
    dense1_units = trial.suggest_int('dense1_units', 32, 128, step=32)
    dropout_rate = trial.suggest_float('dropout_rate', 0.3, 0.5, step=0.1)

    model = Sequential([
        Input(shape=settings["input_shape"]),

        Conv2D(filters=conv1_filters, kernel_size=(3, 3), strides=(2, 2), padding='same'),
        ReLU(max_value=settings["max_relu"]),

        Conv2D(filters=conv2_filters, kernel_size=(3, 3), strides=(2, 2), padding='same'),
        ReLU(max_value=settings["max_relu"]),

        Conv2D(filters=conv3_filters, kernel_size=(3, 3), strides=(2, 2), padding='same'),
        ReLU(max_value=settings["max_relu"]),

        Conv2D(filters=conv4_filters, kernel_size=(1, 1), strides=(1, 1), padding='same'),
        ReLU(max_value=settings["max_relu"]),

        Flatten()
    ])

    # Dynamic addition of Dense layers
    model.add(Dense(dense1_units))
    model.add(ReLU(max_value=settings["max_relu"]))
    model.add(Dropout(dropout_rate))

    if num_dense_layers == 2:
        dense2_units = trial.suggest_int('dense2_units', 16, 64, step=16)
        model.add(Dense(dense2_units))
        model.add(ReLU(max_value=settings["max_relu"]))
        model.add(Dropout(dropout_rate))

    model.add(Dense(settings["num_classes"]))

    return model

In [ ]:
def objective(trial):
    print(f"\n--- Starting Trial {trial.number} ---")

    # Model creation
    model = create_dynamic_model(trial, fixed_settings)

    # Hardware Check (Preventive Pruning)
    # We use a subset of the training set for a dummy, ultra-fast calibration
    dummy_calibration = X_train[:1024]

    try:
        # Dummy quantization
        with set_akida_version(AkidaVersion.v1):
            q_model_dummy = quantize(model, qparams=fixed_settings["qparams"], samples=dummy_calibration)
            akida_model_dummy = cnn2snn.convert(q_model_dummy)

            # Mapping on AKD1000 to read memory usage
            device = akida.AKD1000()
            akida_model_dummy.map(device=device)

            ext_memory = akida_model_dummy.external_memory_size

            if ext_memory > 0:
                # The network is too large, we interrupt this trial immediately
                raise optuna.TrialPruned(f"Pruned: Requires {ext_memory} bytes of external memory.")

    except Exception as e:
        # If the conversion fails due to structural incompatibilities, we discard the trial
        if isinstance(e, optuna.TrialPruned):
            raise e
        raise optuna.TrialPruned(f"Pruned due to conversion error: {str(e)}")

    # Training (Only for models that pass the hardware test)
    lr = trial.suggest_float('learning_rate', 3e-4, 1e-3, log=True)
    batch_size = trial.suggest_categorical('batch_size', [64, 128])

    model.compile(
        optimizer=Adam(learning_rate=lr),
        loss=SparseCategoricalCrossentropy(from_logits=True),
        metrics=['accuracy']
    )

    early_stopping = EarlyStopping(
        monitor='val_accuracy',
        patience=fixed_settings["patience"],
        restore_best_weights=True
    )

    history = model.fit(
        X_train, y_train,
        epochs=fixed_settings["epochs"],
        batch_size=batch_size,
        validation_data=(X_val, y_val),
        callbacks=[early_stopping],
        verbose=0
    )

    # Return the best validation accuracy
    best_val_acc = max(history.history['val_accuracy'])
    print(f"--- Trial {trial.number} finished with accuracy: {best_val_acc} ---")

    return best_val_acc

In [ ]:
# Running the study
print("Starting Optuna optimization...")
study = optuna.create_study(direction='maximize')
# Set n_trials based on the time you have available (e.g., 30-50 to start)
study.optimize(objective, n_trials=30)

print("\n--- Best Hyperparameters Found ---")
print(study.best_params)
print(f"Best Val Accuracy (Baseline Keras): {study.best_value * 100:.2f}%")

In [ ]:
best_params = study.best_params

# Define the path
params_dir = os.path.join(BASE_DIR, 'models/big/best_params')
os.makedirs(params_dir, exist_ok=True) # Ensures the folder exists
params_file_path = os.path.join(params_dir, 'best_optuna_params_big.json')

# Save the dictionary to a JSON file
with open(params_file_path, 'w') as json_file:
    json.dump(best_params, json_file, indent=4)

print(f"Success! Best parameters securely saved to:\n{params_file_path}")

### Load optimal parameters and Model definition

In [ ]:
params_file_path = os.path.join(BASE_DIR, 'models/big/best_params/best_optuna_params_big.json')

# Load the file back into a Python dictionary
with open(params_file_path, 'r') as json_file:
    loaded_params = json.load(json_file)

print("Loaded parameters:")
print(loaded_params)

In [ ]:
def create_akida_compatible_baseline(num_classes):
    optimized_model = Sequential([
        Input(shape=fixed_settings["input_shape"]),

        Conv2D(filters=loaded_params['conv1_filters'], kernel_size=(3, 3), strides=(2, 2), padding='same'),
        ReLU(max_value=fixed_settings["max_relu"]),

        Conv2D(filters=loaded_params['conv2_filters'], kernel_size=(3, 3), strides=(2, 2), padding='same'),
        ReLU(max_value=fixed_settings["max_relu"]),

        Conv2D(filters=loaded_params['conv3_filters'], kernel_size=(3, 3), strides=(2, 2), padding='same'),
        ReLU(max_value=fixed_settings["max_relu"]),

        Conv2D(filters=loaded_params['conv4_filters'], kernel_size=(1, 1), strides=(1, 1), padding='same'),
        ReLU(max_value=fixed_settings["max_relu"]),

        Flatten(),

        Dense(loaded_params['dense1_units']),
        ReLU(max_value=fixed_settings["max_relu"]),
        Dropout(loaded_params['dropout_rate']),
    ])

    if loaded_params.get('num_dense_layers', 1) == 2:
        optimized_model.add(Dense(loaded_params['dense2_units']))
        optimized_model.add(ReLU(max_value=fixed_settings["max_relu"]))
        optimized_model.add(Dropout(loaded_params['dropout_rate']))

    optimized_model.add(Dense(fixed_settings["num_classes"]))

    return optimized_model

optimized_model = create_akida_compatible_baseline(num_classes)
LR = loaded_params['learning_rate']

optimized_model.compile(
    optimizer=Adam(learning_rate=LR),
    loss=SparseCategoricalCrossentropy(from_logits=True),
    metrics=['accuracy']
)

optimized_model.summary()

### Training Loop

In [ ]:
# Create the directory to save models if it does not exist
models_dir = os.path.join(BASE_DIR, 'models/big')
checkpoints_dir = os.path.join(models_dir, 'checkpoint')
os.makedirs(checkpoints_dir, exist_ok=True)

model_path = os.path.join(checkpoints_dir, 'final_model_big_optuna.h5')

# Definition of Callbacks
early_stopping = EarlyStopping(
    monitor='val_loss',
    patience=30,
    restore_best_weights=True
)

model_checkpoint = ModelCheckpoint(
    filepath=model_path,
    monitor='val_loss',
    save_best_only=True,
    verbose=1
)

lr_scheduler = ReduceLROnPlateau(
    monitor='val_loss',
    factor=0.5,
    patience=10,
    verbose=1,
    min_lr=1e-7
)

# Training (Training Loop)
EPOCHS = 300
BATCH_SIZE = loaded_params['batch_size']

In [ ]:
print("Starting training...")
history = optimized_model.fit(
    X_train, y_train,
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    validation_data=(X_val, y_val),
    callbacks=[early_stopping, model_checkpoint, lr_scheduler],
    verbose=1
)

### Evaluation on Test Set

In [ ]:
# Evaluation on the Test Set
print("\n--- Evaluation on the Test Set ---")
test_loss, test_accuracy = optimized_model.evaluate(X_test, y_test, verbose=0)
print(f"Test Loss: {test_loss:.4f}")
print(f"Test Accuracy: {test_accuracy*100:.2f}%")

# Visualization of the results
plt.figure(figsize=(12, 4))

# Loss Plot
plt.subplot(1, 2, 1)
plt.plot(history.history['loss'], label='Train Loss')
plt.plot(history.history['val_loss'], label='Val Loss')
plt.title('Loss during training')
plt.xlabel('Epochs')
plt.ylabel('Loss')
plt.legend()

# Accuracy Plot
plt.subplot(1, 2, 2)
plt.plot(history.history['accuracy'], label='Train Accuracy')
plt.plot(history.history['val_accuracy'], label='Val Accuracy')
plt.title('Accuracy during training')
plt.xlabel('Epochs')
plt.ylabel('Accuracy')
plt.legend()

plt.tight_layout()
plt.show()

### Saving the Model

In [ ]:
# Define the file name
final_model_path = os.path.join(models_dir, 'final_model_big.h5')
optimized_model.save(final_model_path)

print(f"Final model successfully saved in:\n{final_model_path}")

## Quantization with QuantizeML

In [ ]:
# Ensure the model has the best weights saved from the checkpoint
model_path_h5 = os.path.join(models_dir, 'final_model_big.h5')
optimized_model.load_weights(model_path_h5)
print(f"Float model weights loaded from: {model_path_h5}")

print("\n--- Evaluation on the Test Set ---")
test_loss, test_accuracy = optimized_model.evaluate(X_test, y_test, verbose=0)
print(f"Test Loss: {test_loss:.4f}")
print(f"Test Accuracy: {test_accuracy*100:.2f}%")


### Optuna
We leverage again Optuna, but now to find the best learning rate and batch size for fine-tuning our model.

In [ ]:
def objective_qat(trial):
    print(f"\n--- Starting QAT Trial {trial.number} ---")

    # Define the search space for QAT
    num_calib_samples = 2048
    qat_lr = trial.suggest_float('qat_learning_rate', 1e-6, 1e-3, log=True)
    qat_batch_size = trial.suggest_categorical('qat_batch_size', [32, 64, 128])

    # Load the optimized base model (Float)
    optimized_model.load_weights(final_model_path)

    # Prepare the calibration samples
    indices = np.random.choice(len(X_train), num_calib_samples, replace=False)
    calibration_data = X_train[indices]

    # Quantization (PTQ)
    with set_akida_version(AkidaVersion.v1):
        quantized_model = quantize(
            optimized_model,
            qparams=fixed_settings["qparams"],
            samples=calibration_data
        )

    # Compile the quantized model for QAT
    quantized_model.compile(
        optimizer=Adam(learning_rate=qat_lr),
        loss=SparseCategoricalCrossentropy(from_logits=True),
        metrics=['accuracy']
    )

    # Callbacks for QAT
    qat_early_stopping = EarlyStopping(
        monitor='val_accuracy',
        patience=10,
        restore_best_weights=True
    )

    reduce_lr = ReduceLROnPlateau(
        monitor='val_loss',
        factor=0.5,
        patience=5,
        min_lr=1e-7
    )

    # Training (QAT)
    history_qat = quantized_model.fit(
        X_train, y_train,
        epochs=2,
        batch_size=qat_batch_size,
        validation_data=(X_val, y_val),
        callbacks=[qat_early_stopping, reduce_lr],
        verbose=2
    )

    # Return the best validation accuracy achieved
    best_qat_val_acc = max(history_qat.history['val_accuracy'])
    print(f"--- QAT Trial {trial.number} finished with accuracy: {best_qat_val_acc:.4f} ---")

    tf.keras.backend.clear_session()

    return best_qat_val_acc

In [ ]:
print("Starting Optuna optimization for QAT...")
qat_study = optuna.create_study(direction='maximize')
qat_study.optimize(objective_qat, n_trials=1)

print("\n--- Best QAT Hyperparameters Found ---")
print(qat_study.best_params)
print(f"Best Val Accuracy (Quantized): {qat_study.best_value * 100:.2f}%")

In [ ]:
best_qat_params = qat_study.best_params

# Define the path
params_dir = os.path.join(BASE_DIR, 'models/big/best_params')
os.makedirs(params_dir, exist_ok=True) # Ensures the folder exists
qat_params_file_path = os.path.join(params_dir, 'best_optuna_qat_params_big.json')

# Save the dictionary to a JSON file
with open(qat_params_file_path, 'w') as json_file:
    json.dump(best_qat_params, json_file, indent=4)

print(f"Success! Best QAT parameters securely saved to:\n{qat_params_file_path}")

### Quantization

In [ ]:
num_calibration_samples = 10240
indices = np.random.choice(len(X_train), num_calibration_samples, replace=False)
calibration_data = X_train[indices]

qparams = QuantizationParams(
    input_weight_bits=8,
    weight_bits=4,
    activation_bits=4,
)

with set_akida_version(AkidaVersion.v1):
    quantized_model = quantize(
        optimized_model,
        qparams=qparams,
        samples=calibration_data,
        epochs=10
    )

print("\n--- Quantized Model Structure ---")
quantized_model.summary()

quantized_model.compile(
    loss=tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True),
    metrics=['accuracy']
)

print("\n--- Quantized Model Evaluation ---")
q_test_loss, q_test_accuracy = quantized_model.evaluate(X_test, y_test, verbose=0)
print(f"Test Loss (Quantized): {q_test_loss:.4f}")
print(f"Test Accuracy (Quantized): {q_test_accuracy*100:.2f}%")

### Load optimal parameters and Finetuning

In [ ]:
qat_params_file_path = os.path.join(BASE_DIR, 'models/big/best_params/best_optuna_qat_params_big.json')

# Load the file back into a Python dictionary
with open(qat_params_file_path, 'r') as json_file:
    loaded_qat_params = json.load(json_file)

print("Loaded QAT parameters:")
print(loaded_qat_params)

In [ ]:
print("\n--- Quantized Model Fine-Tuning (QAT) Alignment Strategy ---")

qat_model_path = os.path.join(checkpoints_dir, 'final_model_big_quantized.h5')
os.makedirs(os.path.dirname(qat_model_path), exist_ok=True)

BEST_LR = loaded_qat_params['qat_learning_rate']
BEST_BATCH_SIZE = loaded_qat_params['qat_batch_size']

qat_checkpoint = ModelCheckpoint(
    filepath=qat_model_path,
    monitor='val_accuracy',
    save_best_only=True,
    save_weights_only=False,
    mode='max',
    verbose=1
)

qat_early_stopping = EarlyStopping(
    monitor='val_accuracy',
    patience=15,
    mode='max',
    restore_best_weights=True,
    verbose=1
)

lr_scheduler = ReduceLROnPlateau(
    monitor='val_loss',
    factor=0.5,
    patience=5,
    min_lr=1e-7,
    verbose=1
)

# Very low learning rate for post-quantization stability
quantized_model.compile(
    optimizer=Adam(learning_rate=BEST_LR),
    loss=SparseCategoricalCrossentropy(from_logits=True),
    metrics=['accuracy']
)

# Training
history_qat = quantized_model.fit(
    X_train, y_train,
    epochs=2,
    batch_size=BEST_BATCH_SIZE,
    validation_data=(X_val, y_val),
    callbacks=[qat_checkpoint, qat_early_stopping, lr_scheduler],
    verbose=1
)

print(f"Fine-tuning completed and best weights saved directly to: {qat_model_path}")

### Evaluation on test set

In [ ]:
print("--- Quantized Model Evaluation ---")
quantized_model.load_weights(qat_model_path)
q_test_loss, q_test_accuracy = quantized_model.evaluate(X_test, y_test, verbose=0)
print(f"Test Loss (Quantized): {q_test_loss:.4f}")
print(f"Test Accuracy (Quantized): {q_test_accuracy*100:.2f}%")

In [ ]:
# Visualization of Fine-Tuning (QAT) results
plt.figure(figsize=(12, 4))

# Loss Plot
plt.subplot(1, 2, 1)
plt.plot(history_qat.history['loss'], label='QAT Train Loss')
plt.plot(history_qat.history['val_loss'], label='QAT Val Loss')
plt.title('Loss during Fine-Tuning (QAT)')
plt.xlabel('Epochs')
plt.ylabel('Loss')
plt.legend()
plt.grid(True, linestyle='--', alpha=0.6)

# Accuracy Plot
plt.subplot(1, 2, 2)
plt.plot(history_qat.history['accuracy'], label='QAT Train Accuracy')
plt.plot(history_qat.history['val_accuracy'], label='QAT Val Accuracy')
plt.title('Accuracy during Fine-Tuning (QAT)')
plt.xlabel('Epochs')
plt.ylabel('Accuracy')
plt.legend()
plt.grid(True, linestyle='--', alpha=0.6)

plt.tight_layout()
plt.show()

### Saving the Finetuned Quantized Model

In [ ]:
# Save the fine-tuned quantized model weights
qat_model_path = os.path.join(models_dir, 'final_model_big_quantized.h5')
quantized_model.save_weights(qat_model_path)
print(f"Fine-tuned weights saved successfully to: {qat_model_path}")

## Convertion to Akida SNN and Evaluation

In [ ]:
# Load the best QAT weights into the quantized model
qat_model_path = os.path.join(models_dir, 'final_model_big_quantized.h5')
quantized_model.load_weights(qat_model_path)
print(f"QAT weights loaded from: {qat_model_path}")

# Verify that the weights are correct
_, acc_check = quantized_model.evaluate(X_test, y_test, verbose=0)
print(f"Quantized accuracy with QAT weights (pre-conversion): {acc_check*100:.2f}%")

# Conversion
with set_akida_version(AkidaVersion.v1):
    akida_model = cnn2snn.convert(quantized_model)

print("Conversion completed.")

# Evaluation
preds = akida_model.predict(X_test)
akida_preds = np.argmax(preds, axis=-1).flatten()
akida_accuracy = np.mean(akida_preds == y_test)
print(f"\nAkida Accuracy: {akida_accuracy * 100:.2f}%")

# HW Mapping
device = akida.AKD1000()
akida_model.map(device=device)
akida_model.summary()

### Saving the Akida Model

In [ ]:
akida_fb_path = os.path.join(models_dir, 'final_model_big_quantized_akida.fb')
akida_model.save(akida_fb_path)

print(f"Akida SNN model saved successfully in .fb format: {akida_fb_path}")